In [1]:
from Stellarator2 import StellaratorTransport
from yancc_wrapper2 import yancc_data
import yancc
import jax.numpy as jnp
import numpy as np
import jax
import MaNTA
import desc
from desc.plotting import plot_comparison
import matplotlib.pyplot as plt
from desc.profiles import SplineProfile
from desc.optimize._constraint_wrappers import ProximalProjection
from desc.objectives import (
    AspectRatio,
    FixBoundaryR,
    FixBoundaryZ,
    FixCurrent,
    FixPsi,
    ForceBalance,
    LinearObjectiveFromUser,
    ObjectiveFunction,
    ObjectiveFromUser,
    RotationalTransform,
    Volume,
)
from desc.grid import Grid, LinearGrid
from desc.geometry import FourierRZToroidalSurface
from desc.equilibrium import Equilibrium, EquilibriaFamily
import desc.io
from desc import set_device
from scipy.constants import mu_0
import os


Registering cpu implementation for operation get_solution
Registering gpu implementation for operation get_solution
Registering cpu implementation for operation get_adjoint_gradients
Registering gpu implementation for operation get_adjoint_gradients
Registering cpu implementation for operation get_g_val
Registering cpu implementation for operation run
Registering cpu implementation for operation run_ss
Using cache directory: /global/cfs/cdirs/mp217/eatocco/__pycache__
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [2]:

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"



In [6]:

st_config = {
    "ParticleSourceCenter": 0.2,
    "ParticleSourceHeight": 0.1,
    "ParticleSourceWidth": 0.4,
    "HeatSourceCenter": 0.1,
    "HeatSourceHeight": 0.2,
    "HeatSourceWidth": 0.4,
    "EdgeTemperature": 0.2,
    "EdgeDensity": 0.4,
    "n0": 0.5,
    "evolveDensity": True,
}
# runner = MaNTA.Runner(st)

rho_upper = 1.0
rtol = 1e-2
atol = 1e-2
# nodes = [0.0,0.5, 0.75, 0.9, 1.0]
npoints = 4
degree = 4
base = 1.6
tau = 100.0
nodes = 1 - 1.0 / np.logspace(1, npoints - 1, base=base, num=npoints - 1)
nodes = np.concatenate(([0], nodes, [1]))
print(nodes)
# # %%
solver_config = {
    "OutputFilename": "stellarator_w7x",
    "Polynomial_degree": degree,
    "Grid_points": nodes,
    "tau": tau,
    "Lower_boundary": 0.0,
    "Upper_boundary": rho_upper,
    "Relative_tolerance": rtol,
    "Absolute_tolerance": [atol],
    "delta_t": 0.5,
    "initialTimestep": 1e-7,
    "MinStepSize": 1e-9,
    "SteadyStateTolerance": 1e-2,
    "restart": False,
    "zeroFlux": True,
    "aggressiveTimesteps": True,
}


config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}


points = MaNTA.getNodes(
    nodes,
    solver_config["Polynomial_degree"],
)


yancc_rho = jnp.array(points)
yancc_ntheta = 17
yancc_nzeta = 33

yancc_res = {"na": 43, "nx": 5}
# Single hydrogen species. Density and temperature gradients are with
# respect to rho

## to allow maximum fl
# Single hydrogen species. Density and temperature gradients are with
# respect to rhoexibility to match manta, we use a spline with the same control points as manta \
# + axis and lcfs
# initial pressure is all zeros, can change this if desired
pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

eq = desc.examples.get("W7-X")

# Reduce the number of modes (not sure if this is a good thing to do)
eq.change_resolution(M=4, N=4, L_grid=len(points))
eq = eq.solve(x_scale="ess")[0]
eq_init = eq.copy()
yancc_wrapper = yancc_data.from_eq(
    points, eq=eq_init, nt=yancc_ntheta, nz=yancc_nzeta, **yancc_res
)



[0.         0.375      0.609375   0.75585938 1.        ]
Building objective: force
Precomputing transforms
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed Psi
Building objective: fixed pressure
Building objective: fixed iota
Building objective: fixed sheet current
Building objective: self_consistency R
Building objective: self_consistency Z
Building objective: lambda gauge
Building objective: axis R self consistency
Building objective: axis Z self consistency
Number of parameters: 604
Number of objectives: 6798

Starting optimization
Using method: lsq-exact
Optimization terminated successfully.
`ftol` condition satisfied. (ftol=1.00e-02)
         Current function value: 1.069e-05
         Total delta_x: 1.308e-01
         Iterations: 6
         Function evaluations: 7
         Jacobian evaluations: 7
                                                                 Start  -->   End
Total (sum of squares):                                      1.390e-01  --

In [4]:

# with jax.log_compiles(True):
st = StellaratorTransport(config, yancc_wrapper=yancc_wrapper)

st.run()


configuring
Successfully created StellaratorTransport object


ERROR: Residual norm at t = 0: inf
ERROR: Residual norm at t = 0: inf
ERROR: Residual norm at t = 0: inf
ERROR: Residual norm at t = 0: 0.3927214983582881
ERROR: Residual norm at t = 0: 0.21402192114528304
ERROR: Residual norm at t = 0: 0.12513141822554105
ERROR: Residual norm at t = 0: 0.08127005535704429
ERROR: Residual norm at t = 0: 0.059877421155056905
ERROR: Residual norm at t = 0: 0.0495238142924254
ERROR: Residual norm at t = 0: 0.044503574938272616
ERROR: Residual norm at t = 0: 0.042049151073103815
ERROR: Residual norm at t = 0: 0.04083881777121997
ERROR: Residual norm at t = 0: 0.039641117952793986
ERROR: Residual norm at t = 0: 0.04023429358626494
ERROR: Residual norm at t = 0: 0.0333954701277135
ERROR: Residual norm at t = 0: 0.024995627499306547
ERROR: Residual norm at t = 0: 0.022005742932637056
ERROR: Residual norm at t = 0: 0.020980410608225045
ERROR: Residual norm at t = 0: 0.020607968946428735
ERROR: Residual norm at t = 0: 0.020459259696045128
ERROR: Residual norm a

INFO: Total HDG degrees of freedom 215
INFO: Configuration done
INFO: Setting initial conditions
INFO: Number of Residual Evaluations due to IDACalcIC: 25
Writing output at 0.5 ( 20 timesteps )


ERROR: Residual norm at t = 0.5: 2.885185512150229e-05
ERROR: Residual norm at t = 1.0167554266443601: 0.34754256237929254


 dy/dt norm inferred from lambdas is 36.0397
Writing output at 1 ( 21 timesteps )


...

 dy/dt norm inferred from lambdas is 15.1503
Writing output at 5 ( 29 timesteps )


ERROR: Residual norm at t = 5: 0.0003064289891958605
ERROR: Residual norm at t = 5.75981577347875: 0.5159391847953682


 dy/dt norm inferred from lambdas is 14.3651


ERROR: Residual norm at t = 5.75981577347875: 1.0417744525405694
ERROR: Residual norm at t = 5.75981577347875: 2.267199983971279
ERROR: Residual norm at t = 5.75981577347875: 1.3725991713188213
ERROR: Residual norm at t = 5.295517888251937: 6.30154447577649
ERROR: Residual norm at t = 5.295517888251937: 0.6985459425172585
ERROR: Residual norm at t = 5.179443416945234: 1.547019788859039
ERROR: Residual norm at t = 5.179443416945234: 0.5453515894803974
ERROR: Residual norm at t = 5.150424799118558: 0.9711483451010302
ERROR: Residual norm at t = 5.150424799118558: 0.5082892285099674
ERROR: Residual norm at t = 5.143170144661889: 0.852734092673412
ERROR: Residual norm at t = 5.143170144661889: 0.49518976388977576
ERROR: Residual norm at t = 5.141356481047722: 0.6546010965363374
ERROR: Residual norm at t = 5.141356481047722: 0.48992583428647435
ERROR: Residual norm at t = 5.1409030651441805: 0.5134927052535548
ERROR: Residual norm at t = 5.1409030651441805: 0.4882735611235925
ERROR: Residua

JaxRuntimeError: INTERNAL: IDASolve could not complete